# Ideal quantum switch SDP validation

This notebook records the implemented validation path for the ideal quantum switch. The convention is Araujo, Branciard, Costa, Feix, Giarmatzi, Brukner, *New Journal of Physics* 17, 102001 (2015): systems `[AI, AO, BI, BO, F]`, with `F = F_target x F_control`, target state `|0>`, and control state `|+>`.

## Validation target

The generalized robustness of the ideal switch should reproduce the published benchmark near `0.5454` within the repository tolerance. The control-dephased switch is used as a negative control and should have robustness near zero.

In [ ]:
from pathlib import Path
import json

from deltawkrel.switch_models import ideal_quantum_switch_process, dephased_switch_process
from deltawkrel.sdp import (
    SWITCH_BENCHMARK_TOLERANCE,
    SWITCH_GENERALIZED_ROBUSTNESS_REFERENCE,
    solve_switch_generalized_robustness,
)

In [ ]:
W_switch = ideal_quantum_switch_process()
W_dephased = dephased_switch_process()

print(W_switch.shape)
print(W_dephased.shape)

In [ ]:
gen = solve_switch_generalized_robustness(W_switch)
dephased = solve_switch_generalized_robustness(W_dephased)

benchmark_error = abs(gen.objective_value - SWITCH_GENERALIZED_ROBUSTNESS_REFERENCE)
benchmark_passed = (
    gen.status in {"optimal", "optimal_inaccurate"}
    and benchmark_error <= SWITCH_BENCHMARK_TOLERANCE
    and dephased.status in {"optimal", "optimal_inaccurate"}
    and abs(dephased.objective_value) <= 1e-6
)

print(f"ideal switch generalized robustness: {gen.objective_value:.6f}")
print(f"reference: {SWITCH_GENERALIZED_ROBUSTNESS_REFERENCE}")
print(f"error: {benchmark_error:.3e}")
print(f"dephased switch robustness: {dephased.objective_value:.3e}")
print(f"benchmark passed: {benchmark_passed}")

In [ ]:
report = {
    "ideal_switch_generalized_robustness": gen.to_dict(),
    "dephased_switch_generalized_robustness": dephased.to_dict(),
    "reference_generalized_robustness": SWITCH_GENERALIZED_ROBUSTNESS_REFERENCE,
    "tolerance": SWITCH_BENCHMARK_TOLERANCE,
    "benchmark_error": benchmark_error,
    "benchmark_passed": benchmark_passed,
}
Path("../results").mkdir(exist_ok=True)
Path("../results/switch_notebook_validation_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False),
    encoding="utf-8",
)

## Interpretation

A passing run supports the E3 repository claim: the ideal quantum switch benchmark is implemented and externally calibrated against the published generalized robustness. This remains an SDP/software validation, not an experimental validation of the DeltaW/K_rel proposal.